In [0]:
%sql
-- DIM_Asteroide
CREATE TABLE IF NOT EXISTS nasa_dw.gold.dim_asteroide
USING DELTA
COMMENT 'Dimensión de asteroides'
AS SELECT
    asteroide_id, nombre, diametro_promedio_km, categoria_tamano,
    es_potencialmente_peligroso, magnitud_absoluta, url_nasa,
    fecha_acercamiento AS fecha_primera_deteccion,
    CAST(NULL AS INT) AS clasificacion_id,
    CURRENT_TIMESTAMP() AS fecha_actualizacion
FROM nasa_dw.silver.neo_clean WHERE 1=0;

MERGE INTO nasa_dw.gold.dim_asteroide AS tgt
USING (
    SELECT
        asteroide_id, nombre, diametro_promedio_km, categoria_tamano,
        es_potencialmente_peligroso, magnitud_absoluta, url_nasa,
        MIN(fecha_acercamiento) AS fecha_primera_deteccion,
        CAST(NULL AS INT) AS clasificacion_id,
        CURRENT_TIMESTAMP()     AS fecha_actualizacion
    FROM nasa_dw.silver.neo_clean
    GROUP BY asteroide_id, nombre, diametro_promedio_km, categoria_tamano,
             es_potencialmente_peligroso, magnitud_absoluta, url_nasa
) AS src
ON tgt.asteroide_id = src.asteroide_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
USE CATALOG nasa_dw;
-- DIM_Astro

CREATE TABLE IF NOT EXISTS gold.dim_astro
USING DELTA
COMMENT 'Dimensión de tipos de cuerpos celestes'
AS SELECT * FROM (
    VALUES
        (1, 'Asteroide', 'Cuerpo rocoso que orbita el Sol'),
        (2, 'Cometa',    'Cuerpo de hielo y roca que forma cola al acercarse al Sol')
) AS t(astro_id, tipo_astro, descripcion);

In [0]:
%sql
USE CATALOG nasa_dw;
-- DIM_Cometa

CREATE OR REPLACE TABLE gold.dim_cometa
USING DELTA
COMMENT 'Dimensión de cometas'
AS
SELECT
    cometa_id,
    nombre,
    2                       AS astro_id,
    periodo,
    magnitud_absoluta,
    MIN(fecha_acercamiento) AS fecha_primera_deteccion,
    CURRENT_TIMESTAMP()     AS fecha_actualizacion
FROM nasa_dw.silver.cometas_clean
GROUP BY cometa_id, nombre, periodo, magnitud_absoluta;

In [0]:
%sql
USE CATALOG nasa_dw;

CREATE TABLE IF NOT EXISTS gold.dim_clasificacion_orbital
USING DELTA
COMMENT 'Dimensión de clasificación orbital de cuerpos celestes'
AS SELECT * FROM (
    VALUES
        (1, 'Asteroide', 'Apolo',          true,  'Cruza la órbita de la Tierra, semieje mayor > 1 AU'),
        (2, 'Asteroide', 'Aten',           true,  'Cruza la órbita de la Tierra, semieje mayor < 1 AU'),
        (3, 'Asteroide', 'Amor',           true,  'No cruza la órbita de la Tierra, pero se acerca'),
        (4, 'Asteroide', 'Main Belt',      false, 'Cinturón principal entre Marte y Júpiter'),
        (5, 'Cometa',    'Corto Período', false, 'Período orbital < 200 años'),
        (6, 'Cometa',    'Largo Período', false, 'Período orbital > 200 años')
) AS t(clasificacion_id, tipo_cuerpo, nombre, es_neo, descripcion);

In [0]:
%sql
-- Dim_impacto
USE CATALOG nasa_dw;

CREATE TABLE IF NOT EXISTS gold.dim_impacto
USING DELTA
COMMENT 'Dimensión de niveles de peligrosidad de acercamientos'
AS SELECT * FROM (
    VALUES
        (1, 'Bajo',      'Distancia segura o velocidad baja',                    10.0,  30000.0),
        (2, 'Moderado',  'Acercamiento notable pero sin riesgo inmediato',        5.0,  50000.0),
        (3, 'Alto',      'Acercamiento cercano con alta velocidad',               2.0,  70000.0),
        (4, 'Crítico',   'Acercamiento muy cercano (menos de 1 distancia lunar)', 0.0, 100000.0)
) AS t(impacto_id, nivel, descripcion, distancia_minima_lu, velocidad_maxima_kmh);

In [0]:
%sql
-- Validar conteos de dimensiones
SELECT 'dim_fecha' AS tabla, COUNT(*) AS registros FROM nasa_dw.gold.dim_fecha
UNION ALL
SELECT 'dim_asteroide', COUNT(*) FROM nasa_dw.gold.dim_asteroide
UNION ALL
SELECT 'dim_cometa', COUNT(*) FROM nasa_dw.gold.dim_cometa
UNION ALL
SELECT 'dim_astro', COUNT(*) FROM nasa_dw.gold.dim_astro
UNION ALL
SELECT 'dim_clasificacion_orbital', COUNT(*) FROM nasa_dw.gold.dim_clasificacion_orbital
UNION ALL
SELECT 'dim_impacto', COUNT(*) FROM nasa_dw.gold.dim_impacto;